# 03 - Full development matrix

Every model except RSF, across the whole development window.

**5 folds x 5 imputations x 9 models = 225 fits**, each scored on its held-out
fold, plus **45 `dev_full` fits** (one per model per imputation) scored on
`dev_full` only. 270 cells, a few hours.

| | Models | Representation | Configuration |
|---|---|---|---|
| Neural | `variant_a`, `variant_b`, `deephit`, `neural_fine_gray`, `dsm` | `embedding` | `selected_configs.json` |
| Classical | `royston_parmar`, `cox`, `fine_gray`, `rp_kdri_epts` | `onehot` | no tuning needed |

Both representations come from the same fitted `FeatureSpec` - identical
information, never re-encoded. That is what makes the comparison fair.

**RSF is not re-run.** It is already banked in `rsf_results.parquet` from
notebook 01 and is concatenated at analysis time only.

**Configurations are read, not chosen.** A model retuned after seeing
development results is no longer honestly tuned, so nothing here adjusts a
hyperparameter.

**Temporal and exploratory are never loaded** - including by this notebook,
whose results may well prompt another modelling decision. They are scored once,
at the very end, after every such decision is locked.

In [1]:
# =====================================================================
# CONFIGURATION - the only cell to edit
# =====================================================================
from pathlib import Path
import os

"""REPO_ROOT    = Path(r"")          # repository root; leave "" to auto-detect
FEATURES_DIR = Path(r"")          # feature-builder output
RESULTS_DIR  = Path(r"")          # where 01 and 02 wrote their results
NFG_PATH     = Path(r"")          # clone of Jeanselme/NeuralFineGray"""

REPO_ROOT    = Path(r"C:\Users\olf3ph\Documents")          # repository root; leave "" to auto-detect
FEATURES_DIR = Path(r"C:\Users\olf3ph\Documents\FEATURES")          # feature-builder output
RESULTS_DIR  = Path(r"C:\Users\olf3ph\Documents\RESULTS")          # benchmark parquet + logs
NFG_PATH     = Path(r"C:\Users\olf3ph\Documents\NeuralFineGray")


T_EVAL      = [1.0, 3.0, 5.0]     # years
FOLDS       = [0, 1, 2, 3, 4]
IMPUTATIONS = [1, 2, 3, 4, 5]

# =====================================================================
# Nothing below this line needs editing.
# =====================================================================
import sys, json, time
import numpy as np
import pandas as pd
import json

_here = Path.cwd()
if not str(REPO_ROOT):
    REPO_ROOT = _here.parent if (_here / "io_helpers.py").exists() else _here
REPO_ROOT = Path(REPO_ROOT).resolve()

# NFG_PATH must be set BEFORE the deep wrappers are asked what is available:
# the baseline reports itself unavailable when it cannot find the clone, and an
# unavailable model is silently OMITTED from the results rather than raising.
# Losing the closest published comparator that way would not surface until the
# results table was read, hours later.
assert str(NFG_PATH), (
    "NFG_PATH is empty. Neural Fine-Gray is the closest published comparator "
    "to this project's model; without it this run quietly omits it.\n"
    "  git clone --recursive https://github.com/Jeanselme/NeuralFineGray.git")
NFG_PATH = Path(NFG_PATH).resolve()
assert (NFG_PATH / "nfg").is_dir(), f"no nfg/ package under {NFG_PATH}"
os.environ["NFG_PATH"] = str(NFG_PATH)

sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import io_helpers as ioh
ioh.add_rp_core_to_path(REPO_ROOT)

FEATURES_DIR = Path(FEATURES_DIR)
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert FEATURES_DIR.exists(), f"FEATURES_DIR does not exist: {FEATURES_DIR}"

from benchmark.deep import available_deep_baselines
_avail = available_deep_baselines(str(NFG_PATH))
for _name, (_ok, _why) in _avail.items():
    print(f"  {_name:18s} {'available' if _ok else 'UNAVAILABLE: ' + _why}")
assert all(ok for ok, _ in _avail.values()), (
    "a deep baseline is unavailable; fix that before spending hours on a run "
    "that would quietly omit it")

ioh.log("configuration", repo=REPO_ROOT, features=FEATURES_DIR,
        results=RESULTS_DIR)
RESULTS_PATH = RESULTS_DIR / "development_results.parquet"
RSF_PATH = RESULTS_DIR / "rsf_results.parquet"
SELECTED_PATH = RESULTS_DIR / "selected_configs.json"

  deephit            available
  neural_fine_gray   available
  dsm                available
[2026-08-24 10:33:25] configuration  repo=C:\Users\olf3ph\Documents  features=C:\Users\olf3ph\Documents\FEATURES  results=C:\Users\olf3ph\Documents\RESULTS


In [2]:
cfgs = json.loads((RESULTS_DIR / "selected_configs.json").read_text())["models"]
tr, te = ioh.load_fold(FEATURES_DIR, 0, 1, "embedding")

for name in ("variant_a", "variant_b"):
    f = ioh.variant_factory(name, tr["n_continuous"], tr["cardinalities"])
    m = f(cfgs[name]["config"], seed=0)
    m.fit(tr["X"], tr["time"], tr["event"])
    v = np.array([e.val_loss for e in m.history])
    print(f"\n{name}: ran {m.ckpt['epochs_run']} epochs "
          f"(max {m.max_epochs}, patience {m.patience})")
    print(f"  best epoch {int(v.argmin())}")
    print(f"  last 5 val: {np.round(v[-5:], 6)}")


variant_a: ran 19 epochs (max 200, patience 15)
  best epoch 3
  last 5 val: [0.950526 0.957361 0.951663 0.968578 0.964924]

variant_b: ran 19 epochs (max 200, patience 15)
  best epoch 3
  last 5 val: [0.9444   0.946497 0.945663 0.95431  0.953061]


In [3]:
from train import TrainConfig, train_model

base = cfgs["variant_b"]["config"]
trials = {
    "as-run (cosine T_max=200)": dict(),
    "plateau scheduler":         dict(scheduler="plateau"),
    "cosine, patience 40":       dict(patience=40),
    "lower LRs":                 dict(),          # see note below
    "cosine T_max=40":  dict(max_epochs=40),
}
for label, over in trials.items():
    f = ioh.variant_factory("variant_b", tr["n_continuous"], tr["cardinalities"])
    cfg = dict(base)
    if label == "lower LRs":
        cfg.update(spline_lr=0.003, encoder_lr=0.0003, head_lr=0.0003)
    m = f(cfg, seed=0)
    for k, v in over.items():
        setattr(m, k, v)
    m.fit(tr["X"], tr["time"], tr["event"])
    v = np.array([e.val_loss for e in m.history])
    print(f"{label:28s} best {v.min():.6f} at epoch {int(v.argmin()):3d} "
          f"of {len(v)}")

as-run (cosine T_max=200)    best 0.930228 at epoch   3 of 19
plateau scheduler            best 0.930229 at epoch   3 of 19
cosine, patience 40          best 0.930228 at epoch   3 of 44
lower LRs                    best 0.930067 at epoch  13 of 29
cosine T_max=40              best 0.930211 at epoch   3 of 19


## 1. The selected configurations

Printed so the run is self-documenting: the results parquet records a seed and
a model name, not the architecture behind them.

**The tuning scores are not comparable across models.** Each is recorded under
its own criterion - held-out log-likelihood for the variants, Neural Fine-Gray
and DSM; integrated Brier for DeepHit, whose training loss mixes likelihood
with a ranking term and is therefore not a likelihood. Beyond that, a Weibull
mixture and a spline cumulative hazard do not share an estimand, so even two
"log-likelihood" figures sit on different scales. They are provenance, not a
comparison. **The comparison is the CIF-based metrics this notebook produces**,
which every model is scored on identically.

In [2]:
assert SELECTED_PATH.exists(), (
    f"{SELECTED_PATH} not found - run 02_tuning.ipynb first")
selected = json.loads(SELECTED_PATH.read_text())

print(f"tuned on fold {selected.get('tuning_fold')} at "
      f"m={selected.get('tuning_imputation')}, budget "
      f"{selected.get('budget')} configurations each")
print(f"created {selected.get('created')}\n")
for name, entry in selected["models"].items():
    print(f"{name:18s} criterion={entry['criterion']:7s} "
          f"score={entry['score']:+.6f}  (not comparable across rows)")
    print(f"{'':18s} {entry['config']}")

NEURAL = ["variant_a", "variant_b", "deephit", "neural_fine_gray", "dsm"]
CLASSICAL = ["royston_parmar", "cox", "fine_gray", "rp_kdri_epts"]
MODELS = NEURAL + CLASSICAL

missing = [m for m in NEURAL if m not in selected["models"]]
assert not missing, (
    f"no tuned configuration for {missing}. Running those on defaults would "
    f"make them weaker comparators than the ones that were tuned.")
print(f"\n{len(MODELS)} models x {len(FOLDS)} folds x {len(IMPUTATIONS)} "
      f"imputations = {len(MODELS) * len(FOLDS) * len(IMPUTATIONS)} CV cells "
      f"+ {len(MODELS) * len(IMPUTATIONS)} dev_full cells")

tuned on fold 0 at m=1, budget 30 configurations each
created 2026-08-21 14:40:40

variant_a          criterion=loglik  score=-0.895184  (not comparable across rows)
                   {'hidden_dims': [512, 256], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.001, 'spline_lr': 0.03}
variant_b          criterion=loglik  score=-0.892377  (not comparable across rows)
                   {'hidden_dims': [256, 128], 'latent_dim': 128, 'dropout': 0.2, 'encoder_lr': 0.001, 'spline_lr': 0.03, 'head_lr': 0.001}
deephit            criterion=ibs     score=-0.053381  (not comparable across rows)
                   {'hidden': [256, 128], 'head': [64], 'dropout': 0.2, 'lr': 0.001, 'batch_size': 128, 'num_durations': 50, 'alpha': 0.5, 'sigma': 0.1}
neural_fine_gray   criterion=loglik  score=-0.910486  (not comparable across rows)
                   {'hidden': [128, 64], 'head': [64], 'dropout': 0.2, 'lr': 0.003, 'batch_size': 512, 'act': 'Tanh'}
dsm                criterion=loglik  score=-0.881652

## 2. Factories

One place where a model name becomes a fitted object. The classical baselines
are handed `spline_columns` explicitly, read from the `FeatureSpec`, rather
than letting the >=20-distinct-values heuristic decide - that gate is a safety
net, and relying on it would silently expand a many-levelled ordinal.

In [3]:
from benchmark import (BenchmarkCell, CoxBaseline, FineGrayBaseline,
                       MinimalReferenceBaseline, RoystonParmarBaseline,
                       coherence_table, run_benchmark)
from benchmark.deep import (DeepConfig, DSMBaseline, DeepHitBaseline,
                            NeuralFineGrayBaseline)

DEEP_CLASSES = {"deephit": DeepHitBaseline,
                "neural_fine_gray": NeuralFineGrayBaseline,
                "dsm": DSMBaseline}
CLASSICAL_CLASSES = {"royston_parmar": RoystonParmarBaseline,
                     "cox": CoxBaseline,
                     "fine_gray": FineGrayBaseline,
                     "rp_kdri_epts": MinimalReferenceBaseline}

REPRESENTATION = {**{m: "embedding" for m in NEURAL},
                  **{m: "onehot" for m in CLASSICAL}}


def make_factory(model, loaded):
    """`factory(seed) -> baseline`, configured for one training set."""
    cfg = dict(selected["models"].get(model, {}).get("config", {}))

    if model in ("variant_a", "variant_b"):
        return lambda seed: ioh.VariantBaseline(
            variant=model, n_continuous=loaded["n_continuous"],
            cardinalities=loaded["cardinalities"], columns=loaded["columns"],
            seed=seed, **cfg)

    if model in DEEP_CLASSES:
        cls = DEEP_CLASSES[model]
        extra = {"path": str(NFG_PATH)} if model == "neural_fine_gray" else {}
        return lambda seed: cls(
            columns=loaded["columns"], n_continuous=loaded["n_continuous"],
            cardinalities=loaded["cardinalities"], seed=seed,
            config=DeepConfig(**cfg), **extra)

    cls = CLASSICAL_CLASSES[model]
    if model == "rp_kdri_epts":
        # The minimal reference keeps only the two composites and selects its
        # own columns, so it takes no spline list.
        return lambda seed: cls(columns=loaded["columns"], seed=seed)
    spline_cols = ioh.spline_columns_for(loaded)
    return lambda seed: cls(columns=loaded["columns"], seed=seed,
                            spline_columns=spline_cols)


# One sanity load: both representations must describe the same patients.
_fit = ioh.fold_sets(FOLDS[0])[0]
_e = ioh.load_set(FEATURES_DIR, _fit, _fit, IMPUTATIONS[0], "embedding")
_o = ioh.load_set(FEATURES_DIR, _fit, _fit, IMPUTATIONS[0], "onehot")
assert np.array_equal(_e["time"], _o["time"]), \
    "embedding and one-hot disagree on follow-up time - not the same patients"
assert np.array_equal(_e["event"], _o["event"]), \
    "embedding and one-hot disagree on event type - not the same patients"
print(ioh.describe(_e))
print(ioh.describe(_o))
_spl = ioh.spline_columns_for(_o)
print(f"\ncontinuous columns given splines ({len(_spl)}): {_spl[:6]} ...")
del _e, _o

dev_fold0_train m1 [embedding]  37,821 rows x 43 cols   cause1 3,525  cause2 4,431  censored 29,865
dev_fold0_train m1 [onehot]  37,821 rows x 104 cols   cause1 3,525  cause2 4,431  censored 29,865

continuous columns given splines (16): ['AGE', 'AGE_DON', 'HGT_CM_CALC', 'HGT_CM_DON_CALC', 'WGT_KG_CALC', 'WGT_KG_DON_CALC'] ...


## 3. Timing probe

One fit per model at full n on fold 0, m = 1, before committing to 270 cells.
Expected from the tuning run: variants and DSM 10-25 s, DeepHit ~26 s, Neural
Fine-Gray ~50 s. If the projection lands far above three hours, find out why
rather than letting it run overnight.

In [4]:
probe_seconds = {}
for model in MODELS:
    rep = REPRESENTATION[model]
    tr, te = ioh.load_fold(FEATURES_DIR, FOLDS[0], IMPUTATIONS[0], rep)
    try:
        t0 = time.time()
        m = make_factory(model, tr)(0)
        m.fit(tr["X"], tr["time"], tr["event"])
        cif = m.predict_cif(te["X"], np.asarray(T_EVAL))
        probe_seconds[model] = time.time() - t0
        print(f"  {model:18s} [{rep:9s}] {ioh.human(probe_seconds[model]):>8s}"
              f"   F1+F2 max {cif.sum(-1).max():.4f}")
    except Exception as exc:
        print(f"  {model:18s} PROBE FAILED: {type(exc).__name__}: {exc}")
    finally:
        del tr, te

n_cells_per_model = len(FOLDS) * len(IMPUTATIONS) + len(IMPUTATIONS)
total = sum(probe_seconds.values()) * n_cells_per_model
print(f"\n{n_cells_per_model} cells per model x {len(MODELS)} models = "
      f"{n_cells_per_model * len(MODELS)} cells")
print(f"PROJECTED TOTAL: {ioh.human(total)}")
if len(probe_seconds) < len(MODELS):
    print("\n!! not every model probed successfully - fix that before running")

  variant_a          [embedding]      16s   F1+F2 max 0.6580
  variant_b          [embedding]      10s   F1+F2 max 0.6455
  deephit            [embedding]      12s   F1+F2 max 0.5177
  neural_fine_gray   [embedding]      40s   F1+F2 max 0.9358
  dsm                [embedding]      20s   F1+F2 max 0.5032
  royston_parmar     [onehot   ]      38s   F1+F2 max 1.0000
  cox                [onehot   ]     5.3m   F1+F2 max 1.0000
  fine_gray          [onehot   ]    10.8m   F1+F2 max 1.0000
  rp_kdri_epts       [onehot   ]       6s   F1+F2 max 0.7100

30 cells per model x 9 models = 270 cells
PROJECTED TOTAL: 9.23h


## 4-5. Build and run, one model at a time

Cells are built and run **per (model, imputation) group** rather than all 270
up front. `BenchmarkCell` holds its arrays, so building the whole matrix first
would keep every fold of every imputation in memory at once - gigabytes of
design matrices, nearly all idle. Grouping bounds it to one imputation's worth,
at the cost of re-reading the feature files per model: seconds, against hours
of fitting.

Ordering by model also makes progress legible. With ~270 cells over hours, a
per-model elapsed line is what distinguishes a stalled run from a slow one.

`resume=True`, so an interruption continues from the parquet. Seeds come from
`derive_seed(model, fold, imputation)` - a hash, not a counter - so a resumed
run reproduces exactly what an uninterrupted one would have produced.

`dev_full` cells use **`fold = -1`, an integer**. `run_benchmark` writes `fold`
as a parquet column; a string there makes it an object column that pyarrow
refuses outright, and would break any later join against the other models'
integer folds.

In [5]:
DEV_FULL_FOLD = -1
model_seconds = {}

for model in MODELS:
    rep = REPRESENTATION[model]
    t_model = time.time()
    ioh.log(f"START {model}", representation=rep, cells=n_cells_per_model)

    for m_imp in IMPUTATIONS:
        cells = []
        for fold in FOLDS:
            tr, te = ioh.load_fold(FEATURES_DIR, fold, m_imp, rep)
            cells.append(BenchmarkCell(
                model_name=model, fold=int(fold), imputation=int(m_imp),
                train=ioh.as_cell_data(tr),
                evaluate_on={"test": ioh.as_cell_data(te)},
                factory=make_factory(model, tr)))

        full = ioh.load_set(FEATURES_DIR, "dev_full", "dev_full", m_imp, rep)
        cells.append(BenchmarkCell(
            model_name=model, fold=DEV_FULL_FOLD, imputation=int(m_imp),
            train=ioh.as_cell_data(full),
            evaluate_on={"dev_full": ioh.as_cell_data(full)},
            factory=make_factory(model, full)))

        # Guards, on every cell, every time.
        for c in cells:
            expected = {"dev_full"} if c.fold == DEV_FULL_FOLD else {"test"}
            assert set(c.evaluate_on) == expected, (
                f"{model} fold={c.fold}: evaluate_on is "
                f"{sorted(c.evaluate_on)}, expected {sorted(expected)}")
            assert isinstance(c.fold, (int, np.integer)), \
                "fold must stay integer-typed or the parquet write fails"
            ioh.assert_not_held_out(*c.evaluate_on)

        run_benchmark(cells, np.asarray(T_EVAL), RESULTS_PATH,
                      resume=True, verbose=True)
        del cells, full

    model_seconds[model] = time.time() - t_model
    ioh.log(f"DONE  {model}", elapsed=ioh.human(model_seconds[model]))

ioh.log("matrix complete", total=ioh.human(sum(model_seconds.values())))
results = pd.read_parquet(RESULTS_PATH)
print(f"\n{len(results):,} records in {RESULTS_PATH.name}")

[2026-08-23 13:41:24] START variant_a  representation=embedding  cells=30
  skip (done): variant_a fold=0 m=1
  skip (done): variant_a fold=1 m=1
  skip (done): variant_a fold=2 m=1
  skip (done): variant_a fold=3 m=1
  skip (done): variant_a fold=4 m=1
  skip (done): variant_a fold=-1 m=1
  skip (done): variant_a fold=0 m=2
  skip (done): variant_a fold=1 m=2
  skip (done): variant_a fold=2 m=2
  skip (done): variant_a fold=3 m=2
  skip (done): variant_a fold=4 m=2
  skip (done): variant_a fold=-1 m=2
  skip (done): variant_a fold=0 m=3
  skip (done): variant_a fold=1 m=3
  skip (done): variant_a fold=2 m=3
  skip (done): variant_a fold=3 m=3
  skip (done): variant_a fold=4 m=3
  skip (done): variant_a fold=-1 m=3
  skip (done): variant_a fold=0 m=4
  skip (done): variant_a fold=1 m=4
  skip (done): variant_a fold=2 m=4
  skip (done): variant_a fold=3 m=4
  skip (done): variant_a fold=4 m=4
  skip (done): variant_a fold=-1 m=4
  skip (done): variant_a fold=0 m=5
  skip (done): variant

## 6. Post-run checks

Read these before anything downstream. RSF is concatenated here - only here -
from notebook 01's output.

In [6]:
rsf = pd.read_parquet(RSF_PATH) if RSF_PATH.exists() else None
if rsf is None:
    print(f"!! {RSF_PATH.name} not found - RSF will be missing from the table")
    combined = results
else:
    combined = pd.concat([results, rsf], ignore_index=True)
    print(f"concatenated {len(rsf):,} RSF records (m=1 only, from notebook 01)")

print("\nCOHERENCE  max |F1 + F2 + S - 1|")
print(coherence_table(combined).to_string())
print("\nRead this by MECHANISM, not as a ranking: zero by construction "
      "(deephit, neural_fine_gray)")
print("and small-by-measurement (royston_parmar, cox, dsm) are different "
      "quantities - see the README.")


print("\nCOHERENCE and MAXIMUM ALLOCATED MASS")
print(results.groupby("model")
      .agg(coherence_worst=("coherence_max", "max"),
           f1f2_max=("f1f2_max", "max"))
      .sort_values("coherence_worst").to_string())


print("\nMAXIMUM ALLOCATED MASS  max(F1 + F2)")
print(combined.groupby("model")["f1f2_max"].max()
      .sort_values().to_string())
print("\nA model can satisfy the coherence check simply by never allocating "
      "enough mass to violate it. Fine-Gray reaches 0.988 out-of-sample here "
      "while exceeding 1 on dev_full - the guarantee is absent, the violation "
      "is data-dependent.")

concatenated 36 RSF records (m=1 only, from notebook 01)

COHERENCE  max |F1 + F2 + S - 1|
                         worst       average  n_records
model                                                  
neural_fine_gray  0.000000e+00  0.000000e+00        180
dsm               2.220446e-15  1.743050e-15        180
cox               8.881784e-15  7.508808e-15        180
deephit           4.284084e-08  4.172325e-08        180
rp_kdri_epts      1.180606e-06  8.169301e-07        180
variant_a         2.188676e-06  9.485226e-07        180
royston_parmar    3.178402e-06  1.683303e-06        180
variant_b         3.877604e-06  1.731513e-06        180
fine_gray         4.138183e-02  6.927502e-03        180
rsf               6.210453e-02  4.288124e-02         36

Read this by MECHANISM, not as a ranking: zero by construction (deephit, neural_fine_gray)
and small-by-measurement (royston_parmar, cox, dsm) are different quantities - see the README.

COHERENCE and MAXIMUM ALLOCATED MASS
            

In [8]:
print("UNUSABLE CELLS  (these are not merely weak - do not pool them)\n")

bad_conv = combined[~combined["converged"].fillna(True)]
if bad_conv.empty:
    print("  convergence : all cells converged")
else:
    print(f"  !! {len(bad_conv)} record(s) from non-converged fits:")
    print(bad_conv[["model", "fold", "imputation", "cause", "horizon"]]
          .drop_duplicates().to_string(index=False))

# The null is the Aalen-Johansen marginal fitted on each fold's TRAINING rows,
# per fold rather than pooled - the folds differ in event mix, and one shared
# threshold would flatter some and penalise others.


null_table = ioh.null_brier_table(FEATURES_DIR, FOLDS, IMPUTATIONS[0], T_EVAL)
checked = ioh.flag_against_null(
    combined.drop(columns=["null_brier", "beats_null"], errors="ignore"),
    null_table)

"""null_table = ioh.null_brier_table(FEATURES_DIR, FOLDS, IMPUTATIONS[0], T_EVAL)
checked = ioh.flag_against_null(combined, null_table)"""

lost = checked[checked["beats_null"] == False]        # noqa: E712
if lost.empty:
    print("\n  vs null     : every scored cell beats the marginal")
else:
    print(f"\n  !! {len(lost)} record(s) NO BETTER THAN THE MARGINAL:")
    print(lost[["model", "fold", "imputation", "cause", "horizon",
                "brier", "null_brier"]].to_string(index=False))
print("\n  (dev_full rows are scored in-sample and have no matching null "
      "fold, so they are left unflagged rather than compared to a number "
      "that does not apply to them.)")

UNUSABLE CELLS  (these are not merely weak - do not pool them)

  convergence : all cells converged

  !! 29 record(s) NO BETTER THAN THE MARGINAL:
         model  fold  imputation  cause  horizon    brier  null_brier
     variant_a     3           2      2      1.0 0.025552    0.025541
           dsm     0           1      1      1.0 0.027867    0.027762
           dsm     1           1      1      1.0 0.026705    0.026652
           dsm     2           1      1      1.0 0.025010    0.025008
           dsm     3           1      1      1.0 0.026942    0.026823
           dsm     4           1      1      1.0 0.024950    0.024508
           dsm     1           2      1      1.0 0.026760    0.026652
           dsm     2           2      1      1.0 0.025245    0.025008
           dsm     3           2      2      1.0 0.025650    0.025541
           dsm     4           2      1      1.0 0.024660    0.024508
           dsm     0           3      1      1.0 0.027956    0.027762
           d

In [9]:
print("CALIBRATION METHOD per model")
print("(an elevated Fine-Gray fallback rate contaminates ICI in exactly the")
print(" way that survives review - the number is still produced, just weaker)\n")
methods = (combined.groupby(["model", "calibration_method"]).size()
           .unstack(fill_value=0))
print(methods.to_string())
if methods.shape[1] > 1:
    share = methods.div(methods.sum(axis=1), axis=0)
    print("\nshare per model:")
    print(share.round(3).to_string())

CALIBRATION METHOD per model
(an elevated Fine-Gray fallback rate contaminates ICI in exactly the
 way that survives review - the number is still produced, just weaker)

calibration_method  fine_gray_rcs
model                            
cox                           180
deephit                       180
dsm                           180
fine_gray                     180
neural_fine_gray              180
royston_parmar                180
rp_kdri_epts                  180
rsf                            36
variant_a                     180
variant_b                     180


In [10]:
print("WALL TIME per model\n")
# fit + predict, because RSFBaseline defers ALL its work to a single R
# subprocess at predict time: its fit_seconds reads 0.0, and a table built on
# that column alone would say the bottleneck of the whole project was free.
per_cell = (combined.groupby(["model", "fold", "imputation", "eval_set"])
            [["fit_seconds", "predict_seconds"]].first().reset_index())
per_cell["total_seconds"] = per_cell["fit_seconds"] + per_cell["predict_seconds"]
timing = (per_cell.groupby("model")["total_seconds"]
          .agg(["sum", "mean", "count"])
          .rename(columns={"sum": "total_s", "mean": "per_cell_s",
                           "count": "cells"})
          .sort_values("total_s", ascending=False))
timing["total"] = timing["total_s"].map(ioh.human)
print(timing[["cells", "per_cell_s", "total"]].round(1).to_string())
print(f"\ngrand total (fit + predict): "
      f"{ioh.human(per_cell['total_seconds'].sum())}")
print("RSF's fit_seconds is 0.0 by construction - it builds the forest at "
      "predict time - so the column above is fit + predict for every model.")

WALL TIME per model

                  cells  per_cell_s  total
model                                     
fine_gray            30       707.9  5.90h
cox                  30       312.9  2.61h
neural_fine_gray     30        43.7  21.8m
royston_parmar       30        37.0  18.5m
rsf                   6       128.5  12.8m
dsm                  30        24.3  12.1m
variant_a            30        20.0  10.0m
deephit              30        13.7   6.9m
variant_b            30        13.3   6.7m
rp_kdri_epts         30         5.2   2.6m

grand total (fit + predict): 10.03h
RSF's fit_seconds is 0.0 by construction - it builds the forest at predict time - so the column above is fit + predict for every model.


## 7. Headline table

**Calibration first.** The endpoint is calibrated absolute risk, not
discrimination: a model can rank patients well and still be wrong about how
much risk each one carries, which is the failure this whole exercise exists to
detect. AUC is reported, but it is not the result. Rows are ordered by ICI.

Mean +/- SD across folds and imputations, per model x cause x horizon. The SD
mixes fold and imputation variance, so read it as spread rather than as a
standard error.

In [11]:
METRICS = ["ici", "e50", "e90", "brier", "auc", "coherence_max"]
cv = combined[combined["eval_set"] == "test"]

summary = (cv.groupby(["model", "cause", "horizon"])[METRICS]
           .agg(["mean", "std"]))
summary.columns = [f"{a}_{b}" for a, b in summary.columns]


def _pm(mu, sd):
    if not np.isfinite(mu):
        return "n/a"
    return f"{mu:.4f} +/- {sd:.4f}" if np.isfinite(sd) else f"{mu:.4f}"


for cause in sorted(cv["cause"].unique()):
    for horizon in sorted(cv["horizon"].unique()):
        block = summary.xs((cause, horizon), level=("cause", "horizon"))
        print(f"\n=== cause {int(cause)}, {horizon:.0f} years   "
              f"(mean +/- SD over {cv['fold'].nunique()} folds x "
              f"{cv['imputation'].nunique()} imputations)")
        rows = []
        for model in block.index:
            row = {"model": model}
            for metric in METRICS:
                row[metric] = _pm(block.loc[model, f"{metric}_mean"],
                                  block.loc[model, f"{metric}_std"])
            rows.append(row)
        table = pd.DataFrame(rows)
        # Sorted by ICI: calibration is the endpoint, so it orders the table.
        order = block["ici_mean"].sort_values().index
        table = table.set_index("model").loc[order].reset_index()
        print(table.to_string(index=False))

summary.to_csv(RESULTS_DIR / "development_summary.csv")
ioh.log("summary written", path=RESULTS_DIR / "development_summary.csv")


=== cause 1, 1 years   (mean +/- SD over 5 folds x 5 imputations)
           model               ici               e50               e90             brier               auc     coherence_max
       fine_gray 0.0023 +/- 0.0005 0.0014 +/- 0.0008 0.0038 +/- 0.0007 0.0261 +/- 0.0012 0.6234 +/- 0.0085 0.0001 +/- 0.0007
             cox 0.0024 +/- 0.0005 0.0014 +/- 0.0008 0.0039 +/- 0.0009 0.0260 +/- 0.0012 0.6294 +/- 0.0085 0.0000 +/- 0.0000
    rp_kdri_epts 0.0024 +/- 0.0008 0.0022 +/- 0.0010 0.0035 +/- 0.0012 0.0261 +/- 0.0012 0.5966 +/- 0.0070 0.0000 +/- 0.0000
             rsf 0.0025 +/- 0.0010 0.0019 +/- 0.0012 0.0051 +/- 0.0027 0.0260 +/- 0.0013 0.6377 +/- 0.0167 0.0390 +/- 0.0057
  royston_parmar 0.0027 +/- 0.0006 0.0017 +/- 0.0004 0.0044 +/- 0.0015 0.0260 +/- 0.0012 0.6293 +/- 0.0085 0.0000 +/- 0.0000
       variant_a 0.0033 +/- 0.0014 0.0027 +/- 0.0014 0.0057 +/- 0.0022 0.0260 +/- 0.0012 0.6233 +/- 0.0153 0.0000 +/- 0.0000
       variant_b 0.0040 +/- 0.0017 0.0028 +/- 0.0014 0.007

## 8. Summary

**Produced**

- `development_results.parquet` - long format, one row per (model, fold,
  imputation, eval_set, cause, horizon), with AUC, C-index, Brier, Austin
  calibration (ICI/E50/E90/Emax), calibration-in-the-large, the coherence
  residual, timings, seeds, and a per-cause `converged` flag. `fold = -1`
  marks the full-development fits.
- `development_summary.csv` - the headline table, mean and SD across folds and
  imputations.

**Covers** nine models across 5 folds x 5 imputations, plus a full-development
fit per model per imputation. RSF is *not* in this file; it lives in
`rsf_results.parquet` at m = 1 and is concatenated at analysis time.

**Temporal and exploratory remain untouched.**

**Consumed by** the final evaluation, which locks the model choice made here
and scores it once on the held-out windows. That notebook is the only place
those windows are loaded, and it should not be written until every decision
this table prompts has been made.

**To report in the methods:** the selected configurations (cell 1), the
calibration-method shares (cell 6), RSF's single-imputation deviation, and
whatever this table changes about the model choice.

In [12]:
print(f"development_results.parquet : {RESULTS_PATH}")
print(f"development_summary.csv     : {RESULTS_DIR / 'development_summary.csv'}")
print(f"rsf_results.parquet         : {RSF_PATH} "
      f"({'present' if RSF_PATH.exists() else 'MISSING'})")
print(f"\nrecords     : {len(results):,}"
      f" (+{0 if rsf is None else len(rsf):,} RSF, concatenated at analysis)")
print(f"models      : {sorted(results.model.unique())}")
print(f"folds       : {sorted(results.fold.unique())}  (-1 = dev_full)")
print(f"imputations : {sorted(results.imputation.unique())}")
print(f"eval sets   : {sorted(results.eval_set.unique())}")
print(f"wall time   : {ioh.human(sum(model_seconds.values()))}")

development_results.parquet : C:\Users\olf3ph\Documents\RESULTS\development_results.parquet
development_summary.csv     : C:\Users\olf3ph\Documents\RESULTS\development_summary.csv
rsf_results.parquet         : C:\Users\olf3ph\Documents\RESULTS\rsf_results.parquet (present)

records     : 1,620 (+36 RSF, concatenated at analysis)
models      : ['cox', 'deephit', 'dsm', 'fine_gray', 'neural_fine_gray', 'royston_parmar', 'rp_kdri_epts', 'variant_a', 'variant_b']
folds       : [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]  (-1 = dev_full)
imputations : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
eval sets   : ['dev_full', 'test']
wall time   : 17s
